In [1]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import duckdb

# Grouping by day

In [ ]:
# Initialize DuckDB connection
con = duckdb.connect()

# Load join dates (small table, can keep in memory)
user_of_interests = "../../data/ale_simplicistic_model/relative/filtered/first_week_blockers.parquet"
con.execute("CREATE TABLE join_dates AS SELECT * FROM read_parquet(?)", [user_of_interests])

print("Join dates loaded")

Join dates loaded


In [3]:
# Helper: build 7-day vectors using DuckDB for efficient large-table processing
def make_vec_duckdb(file_path, left_on='did_id', vec_name='vec'):
    """
    Use DuckDB to:
    1. Read parquet file
    2. Join with join_dates
    3. Compute days_since_join
    4. Filter to first week (0-6 days)
    5. Aggregate counts per user per day
    6. Pivot to create 7-element vectors
    """
    query = f"""
    WITH activity AS (
        SELECT 
            act.{left_on} as id_col,
            act.created_date,
            jd.join_date
        FROM read_parquet('{file_path}') act
        LEFT JOIN join_dates jd ON act.{left_on} = jd.did_id
    ),
    days_computed AS (
        SELECT 
            id_col,
            CAST(ROUND((EPOCH(created_date::TIMESTAMP) - EPOCH(join_date::TIMESTAMP)) / 86400.0) AS INTEGER) AS days_since_join
        FROM activity
        WHERE join_date IS NOT NULL
    ),
    first_week AS (
        SELECT id_col, days_since_join
        FROM days_computed
        WHERE days_since_join >= 0 AND days_since_join <= 6
    ),
    counts AS (
        SELECT 
            id_col,
            days_since_join,
            COUNT(*) as cnt
        FROM first_week
        GROUP BY id_col, days_since_join
    ),
    pivoted AS (
        SELECT
            id_col,
            MAX(CASE WHEN days_since_join = 0 THEN cnt ELSE 0 END) AS d0,
            MAX(CASE WHEN days_since_join = 1 THEN cnt ELSE 0 END) AS d1,
            MAX(CASE WHEN days_since_join = 2 THEN cnt ELSE 0 END) AS d2,
            MAX(CASE WHEN days_since_join = 3 THEN cnt ELSE 0 END) AS d3,
            MAX(CASE WHEN days_since_join = 4 THEN cnt ELSE 0 END) AS d4,
            MAX(CASE WHEN days_since_join = 5 THEN cnt ELSE 0 END) AS d5,
            MAX(CASE WHEN days_since_join = 6 THEN cnt ELSE 0 END) AS d6
        FROM counts
        GROUP BY id_col
    )
    SELECT 
        id_col AS did_id,
        [d0, d1, d2, d3, d4, d5, d6] AS {vec_name}
    FROM pivoted
    """
    
    return con.execute(query).df()

print("DuckDB helper function defined")

DuckDB helper function defined


In [ ]:
# Posts vectors using DuckDB (no intermediate posts_vec_fixed)
posts_path = "../../data/ale_simplicistic_model/relative/filtered/posts.parquet"
posts_vec_df = make_vec_duckdb(posts_path, left_on='did_id', vec_name='posts_vec')

# Register the pandas DataFrame so DuckDB can reference it by name
con.register("posts_vec_df", posts_vec_df)

# Create uoi_activity with posts_vec already COALESCED to zeros when missing
con.execute("""
CREATE TABLE uoi_activity AS
SELECT jd.*,
       COALESCE(pv.posts_vec, [0,0,0,0,0,0,0]) AS posts_vec
FROM join_dates jd
LEFT JOIN posts_vec_df pv ON jd.did_id = pv.did_id
""")

print("Posts processed:")
print(con.execute("SELECT did_id, posts_vec FROM uoi_activity LIMIT 5").df())

Posts processed:
     did_id                     posts_vec
0  32373504       [13, 5, 3, 5, 6, 9, 13]
1   8429753  [12, 28, 50, 61, 57, 49, 36]
2  22954618         [3, 2, 4, 8, 5, 0, 1]
3  15581588       [0, 0, 0, 5, 0, 15, 25]
4  14545846      [20, 9, 21, 3, 8, 8, 12]


In [ ]:
# Blocks: actor and subject 7-day vectors using DuckDB
blocks_path = "../../data/ale_simplicistic_model/relative/filtered/blocks.parquet"

# Actor vectors
blocks_actor_vec = make_vec_duckdb(blocks_path, left_on='did_id', vec_name='blocks_actor_vec')
con.execute("""
    CREATE OR REPLACE TABLE uoi_activity AS 
    SELECT 
        ua.*,
        COALESCE(bav.blocks_actor_vec, [0,0,0,0,0,0,0]) AS blocks_actor_vec
    FROM uoi_activity ua
    LEFT JOIN blocks_actor_vec bav ON ua.did_id = bav.did_id
""")

# Subject vectors
blocks_subject_vec = make_vec_duckdb(blocks_path, left_on='subject_did_id', vec_name='blocks_subject_vec')
con.execute("""
    CREATE OR REPLACE TABLE uoi_activity AS 
    SELECT 
        ua.*,
        COALESCE(bsv.blocks_subject_vec, [0,0,0,0,0,0,0]) AS blocks_subject_vec
    FROM uoi_activity ua
    LEFT JOIN blocks_subject_vec bsv ON ua.did_id = bsv.did_id
""")

print("Blocks processed:")
print(con.execute("SELECT did_id, blocks_actor_vec, blocks_subject_vec FROM uoi_activity LIMIT 5").df())

Blocks processed:
     did_id        blocks_actor_vec     blocks_subject_vec
0  32373504  [3, 0, 0, 0, 13, 0, 6]  [0, 0, 0, 0, 0, 0, 1]
1  15581588   [0, 0, 0, 0, 0, 1, 0]  [0, 0, 0, 2, 0, 0, 0]
2  11534330   [0, 0, 0, 0, 0, 0, 1]  [0, 0, 1, 0, 1, 0, 0]
3   6952429   [0, 0, 2, 1, 1, 0, 0]  [0, 0, 1, 2, 0, 0, 1]
4  29163033   [3, 0, 0, 0, 0, 0, 1]  [1, 0, 0, 0, 0, 2, 0]


In [ ]:
# Follows: actor and subject vectors using DuckDB (handles large tables efficiently)
follows_path = "../../data/ale_simplicistic_model/relative/filtered/follows.parquet"

# Actor vectors
follows_actor_vec = make_vec_duckdb(follows_path, left_on='did_id', vec_name='follows_actor_vec')
con.execute("""
    CREATE OR REPLACE TABLE uoi_activity AS 
    SELECT 
        ua.*,
        COALESCE(fav.follows_actor_vec, [0,0,0,0,0,0,0]) AS follows_actor_vec
    FROM uoi_activity ua
    LEFT JOIN follows_actor_vec fav ON ua.did_id = fav.did_id
""")

# Subject vectors
follows_subject_vec = make_vec_duckdb(follows_path, left_on='subject_did_id', vec_name='follows_subject_vec')
con.execute("""
    CREATE OR REPLACE TABLE uoi_activity AS 
    SELECT 
        ua.*,
        COALESCE(fsv.follows_subject_vec, [0,0,0,0,0,0,0]) AS follows_subject_vec
    FROM uoi_activity ua
    LEFT JOIN follows_subject_vec fsv ON ua.did_id = fsv.did_id
""")

print("Follows processed:")
print(con.execute("SELECT did_id, follows_actor_vec, follows_subject_vec FROM uoi_activity LIMIT 5").df())

Follows processed:
     did_id              follows_actor_vec          follows_subject_vec
0  32373504      [32, 8, 4, 2, 54, 15, 46]       [10, 5, 4, 1, 1, 9, 8]
1  15581588          [0, 0, 0, 1, 0, 0, 5]        [0, 0, 0, 1, 0, 0, 4]
2  11534330     [59, 58, 2, 0, 3, 13, 105]        [5, 6, 0, 0, 0, 5, 8]
3   6952429  [28, 23, 330, 87, 46, 26, 23]  [8, 16, 68, 68, 52, 19, 17]
4  29163033        [21, 28, 1, 1, 3, 3, 6]        [1, 3, 3, 0, 1, 0, 3]


In [ ]:
# Likes: actor and subject vectors using DuckDB
likes_path = "../../data/ale_simplicistic_model/relative/filtered/likes.parquet"

# Actor vectors
likes_actor_vec = make_vec_duckdb(likes_path, left_on='did_id', vec_name='likes_actor_vec')
con.execute("""
    CREATE OR REPLACE TABLE uoi_activity AS 
    SELECT 
        ua.*,
        COALESCE(lav.likes_actor_vec, [0,0,0,0,0,0,0]) AS likes_actor_vec
    FROM uoi_activity ua
    LEFT JOIN likes_actor_vec lav ON ua.did_id = lav.did_id
""")

# Subject vectors
likes_subject_vec = make_vec_duckdb(likes_path, left_on='subject_did_id', vec_name='likes_subject_vec')
con.execute("""
    CREATE OR REPLACE TABLE uoi_activity AS 
    SELECT 
        ua.*,
        COALESCE(lsv.likes_subject_vec, [0,0,0,0,0,0,0]) AS likes_subject_vec
    FROM uoi_activity ua
    LEFT JOIN likes_subject_vec lsv ON ua.did_id = lsv.did_id
""")

print("Likes processed:")
print(con.execute("SELECT did_id, likes_actor_vec, likes_subject_vec FROM uoi_activity LIMIT 5").df())

Likes processed:
     did_id               likes_actor_vec           likes_subject_vec
0  32373504  [49, 33, 32, 20, 32, 47, 82]       [5, 0, 0, 1, 0, 2, 2]
1  15581588         [1, 0, 0, 2, 0, 2, 6]       [0, 0, 0, 1, 0, 1, 9]
2  11534330     [14, 37, 7, 0, 3, 20, 62]       [3, 2, 0, 0, 0, 1, 6]
3   6952429  [10, 17, 42, 28, 46, 20, 18]  [4, 5, 21, 35, 47, 29, 57]
4  29163033      [4, 16, 3, 6, 6, 10, 18]      [1, 4, 0, 1, 2, 3, 12]


## Save the file

In [ ]:
# Export final result to parquet using DuckDB
save_path = "../../data/ale_simplicistic_model/relative/processed/user_activity.parquet"

# Get final dataframe and save
final_df = con.execute("SELECT * FROM uoi_activity").df()

# Convert to pyarrow and save (DuckDB list types are compatible with pyarrow)
arrow_table = pa.Table.from_pandas(final_df, preserve_index=False)
pq.write_table(arrow_table, save_path, compression='zstd')

print('Saved user_table to', save_path)
print('Number of rows:', len(final_df))
print('Columns:', list(final_df.columns))
print('\nSample of final data:')
print(final_df.head())

# Close DuckDB connection
con.close()

Saved user_table to ../data/ale_simplicistic_model/processed/user_activity.parquet
Number of rows: 100000
Columns: ['did_id', 'join_date', 'posts_vec', 'blocks_actor_vec', 'blocks_subject_vec', 'follows_actor_vec', 'follows_subject_vec', 'likes_actor_vec', 'likes_subject_vec']

Sample of final data:
     did_id  join_date                    posts_vec        blocks_actor_vec  \
0  32373504 2024-09-02      [13, 5, 3, 5, 6, 9, 13]  [3, 0, 0, 0, 13, 0, 6]   
1  15581588 2024-10-18      [0, 0, 0, 5, 0, 15, 25]   [0, 0, 0, 0, 0, 1, 0]   
2  11534330 2024-11-10       [1, 2, 1, 0, 0, 4, 14]   [0, 0, 0, 0, 0, 0, 1]   
3   6952429 2024-11-08  [5, 11, 30, 21, 28, 18, 15]   [0, 0, 2, 1, 1, 0, 0]   
4  29163033 2025-01-11        [9, 8, 1, 2, 1, 6, 4]   [3, 0, 0, 0, 0, 0, 1]   

      blocks_subject_vec              follows_actor_vec  \
0  [0, 0, 0, 0, 0, 0, 1]      [32, 8, 4, 2, 54, 15, 46]   
1  [0, 0, 0, 2, 0, 0, 0]          [0, 0, 0, 1, 0, 0, 5]   
2  [0, 0, 1, 0, 1, 0, 0]     [59, 58, 2, 0, 3, 